In [ ]:
code = 'DTN'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def DTN(bt, start_time, end_time, orderside, method, decay, sl, om, target):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt

        # CE Trade
        ce_open, ce_high, ce_low, ce_close, ce_decay_price, ce_decay_flag, ce_decay_time = bt.decay_check_single_leg(start_dt, end_dt, ce_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)

        ce_re_decay_price, ce_re_decay_flag, ce_re_decay_time = '', False, ''
        ce_re_sl_price, ce_re_sl_flag, ce_re_sl_time, ce_re_pnl = '', False, '', 0
        
        if ce_decay_flag:
            ce_target_price = (ce_decay_price - cal_percent(ce_decay_price, target)) if orderside == 'SELL' else (ce_decay_price + cal_percent(ce_decay_price, target))
            ce_sl_price, ce_sl_flag, _, ce_target_flag, ce_sl_time, ce_pnl = bt.sl_check_single_leg(ce_decay_time, end_dt, ce_scrip, o=(None if method == 'CC' else ce_decay_price), sl=sl, target_price=ce_target_price, orderside=orderside, from_candle_close=from_candle_close)
        
            if ce_target_flag:
                ce_re_decay_price, ce_re_decay_flag, ce_re_decay_time = bt.decay_check_single_leg(ce_sl_time, end_dt, ce_scrip, decay_price=ce_decay_price, from_candle_close=from_candle_close, orderside = ('BUY' if orderside == 'SELL' else 'SELL'))
                
                if ce_re_decay_flag:
                    ce_re_sl_price, ce_re_sl_flag, _, _, ce_re_sl_time, ce_re_pnl = bt.sl_check_single_leg(ce_re_decay_time, end_dt, ce_scrip, o=(None if method == 'CC' else ce_decay_price), sl=sl, target_price=ce_target_price, orderside=orderside, from_candle_close=from_candle_close)
        else:
            ce_sl_price, ce_sl_flag, ce_target_price, ce_target_flag, ce_sl_time, ce_pnl = '', False, '', False, '', 0
            
        ce_first_entrie = [ce_scrip, ce_price, ce_decay_price, ce_decay_flag, ce_decay_time, ce_sl_price, ce_target_price, ce_target_flag, ce_sl_flag, ce_sl_time, ce_pnl]
        ce_re_entrie = [ce_re_decay_price, ce_re_decay_flag, ce_re_decay_time, ce_re_sl_price, ce_re_sl_flag, ce_re_sl_time, ce_re_pnl]
        
        # PE Trade
        pe_open, pe_high, pe_low, pe_close, pe_decay_price, pe_decay_flag, pe_decay_time = bt.decay_check_single_leg(start_dt, end_dt, pe_scrip, decay=decay, from_candle_close=from_candle_close, orderside=orderside, with_ohlc=True)
        
        pe_re_decay_price, pe_re_decay_flag, pe_re_decay_time = '', False, ''
        pe_re_sl_price, pe_re_sl_flag, pe_re_sl_time, pe_re_pnl = '', False, '', 0
        
        if pe_decay_flag:
            pe_target_price = (pe_decay_price - cal_percent(pe_decay_price, target)) if orderside == 'SELL' else (pe_decay_price + cal_percent(pe_decay_price, target))
            pe_sl_price, pe_sl_flag, _, pe_target_flag, pe_sl_time, pe_pnl = bt.sl_check_single_leg(pe_decay_time, end_dt, pe_scrip, o=(None if method == 'CC' else pe_decay_price), sl=sl, target_price=pe_target_price, orderside=orderside, from_candle_close=from_candle_close)
        
            if pe_target_flag:
                pe_re_decay_price, pe_re_decay_flag, pe_re_decay_time = bt.decay_check_single_leg(pe_sl_time, end_dt, pe_scrip, decay_price=pe_decay_price, from_candle_close=from_candle_close, orderside = ('BUY' if orderside == 'SELL' else 'SELL'))
                
                if pe_re_decay_flag:
                    pe_re_sl_price, pe_re_sl_flag, _, _, pe_re_sl_time, pe_re_pnl = bt.sl_check_single_leg(pe_re_decay_time, end_dt, pe_scrip, o=(None if method == 'CC' else pe_decay_price), sl=sl, target_price=pe_target_price, orderside=orderside, from_candle_close=from_candle_close)
        else:
            pe_sl_price, pe_sl_flag, pe_target_price, pe_target_flag, pe_sl_time, pe_pnl = '', False, '', False, '', 0

        pe_first_entrie = [pe_scrip, pe_price, pe_decay_price, pe_decay_flag, pe_decay_time, pe_sl_price, pe_target_price, pe_target_flag, pe_sl_flag, pe_sl_time, pe_pnl]
        pe_re_entrie = [pe_re_decay_price, pe_re_decay_flag, pe_re_decay_time, pe_re_sl_price, pe_re_sl_flag, pe_re_sl_time, pe_re_pnl]

        total = ce_pnl + ce_re_pnl + pe_pnl + pe_re_pnl
        return [code, bt.index, start_time, end_time, orderside, method, decay, sl, om, target, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + ce_first_entrie + ce_re_entrie + pe_first_entrie + pe_re_entrie + [total]
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, decay, sl, om, target])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re, re_entries = 7, 7

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_Decay/P_SL/P_OM/P_Target/Date/Day/DTE/EntryTime/Future')
            
            ce_log = f'/CE.Strike/CE.Price/CE.Decay.Price/CE.Decay.Flag/CE.Decay.Time/CE.SL.Price/CE.Target.Price/CE.Target.Flag/CE.SL.Flag/CE.Exit.Time/CE.PNL/CE.RE.Decay.Price/CE.RE.Decay.Flag/CE.RE.Decay.Time/CE.RE.SL.Price/CE.RE.SL.Flag/CE.RE.Exit.Time/CE.RE.PNL'
            pe_log = f'/PE.Strike/PE.Price/PE.Decay.Price/PE.Decay.Flag/PE.Decay.Time/PE.SL.Price/PE.Target.Price/PE.Target.Flag/PE.SL.Flag/PE.Exit.Time/PE.PNL/PE.RE.Decay.Price/PE.RE.Decay.Flag/PE.RE.Decay.Time/PE.RE.SL.Price/PE.RE.SL.Flag/PE.RE.Exit.Time/PE.RE.PNL'

            log_cols = log_cols + ce_log + pe_log + '/Total.PNL'
            log_cols = log_cols.split('/')
            
            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [DTN(bt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['decay'], row['sl'], row['om'], row['target']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))